# Curso: Bioestatística — Regressão Prais-Winsten
## Autores: Sandro Camargo e Fernando Cardoso

**Problema**:
Comparar o crescimento de bovinos de diferentes genótipos, 24 novilhos de vacas Angus inseminadas com touros de três raças zebuı́nas: Brahman, Nelore e Tabapuã;

Os novilhos foram criados em condições extensivas,
exclusivamente sob pastagem nativa, em campos
experimentais da Embrapa Pecuária Sul (Bagé/RS);

Os novilhos foram pesados em cinco perı́odos:
* Desmama (∼ 6 meses de idade)
* Ano (∼ 12 meses)
* Sobreano (∼ 18 meses)
* Ao final da recria (∼ 36 meses)
* Antes do abate (∼ 39 meses)

A base de dados está disponível [aqui](https://github.com/Sandrocamargo/biostatistics/blob/master/datasets/medidasrepetidas-cruzas.csv).

Abra este código no seu google colab [clicando aqui](https://colab.research.google.com/github/Sandrocamargo/biostatistics/blob/master/python/bioe_09_Prais_Winsten.ipynb).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson

In [ ]:
# ---------------------------
# 1. Carga e Preparação dos Dados
# ---------------------------
plt.style.use("seaborn-v0_8-white")
DATA_PATH = "https://raw.githubusercontent.com/Sandrocamargo/biostatistics/refs/heads/master/datasets/medidasrepetidas-cruzas.csv"

dados = pd.read_csv(DATA_PATH, sep=";", header=0)
dados["Datad"] = pd.to_datetime(dados["Data"], errors="coerce")

# Ordenar cronologicamente e criar variável numérica de tempo (dias desde a 1ª pesagem)
dados = dados.sort_values("Datad").reset_index(drop=True)
dados["Datad_num"] = (dados["Datad"] - dados["Datad"].min()).dt.days

In [ ]:
# ---------------------------
# 2. Função de Regressão Prais-Winsten (FGLS Iterativo com retenção de t=1)
# ---------------------------
def prais_winsten(y, X, max_iter=100, tol=1e-6):
    """
    Estima regressão de Prais-Winsten para autocorrelação AR(1).
    Mantém a primeira observação transformada por sqrt(1 - rho^2).
    """
    y = np.asarray(y, dtype=float)
    X = np.asarray(X, dtype=float)
    n, k = X.shape

    # Passo 1: Ajuste OLS inicial
    ols_model = sm.OLS(y, X).fit()
    resid = ols_model.resid
    dw_inicial = durbin_watson(resid)

    rho = 0.0
    for _ in range(max_iter):
        rho_prev = rho
        # Estimação de rho (autocorrelação de 1ª ordem dos resíduos)
        rho = np.sum(resid[1:] * resid[:-1]) / np.sum(resid[:-1] ** 2)
        rho = np.clip(rho, -0.99, 0.99)

        # Transformação de Prais-Winsten
        y_star = np.empty_like(y)
        X_star = np.empty_like(X)

        # Primeira observação (preserva graus de liberdade)
        y_star[0] = np.sqrt(1 - rho**2) * y[0]
        X_star[0, :] = np.sqrt(1 - rho**2) * X[0, :]

        # Demais observações (t >= 2)
        y_star[1:] = y[1:] - rho * y[:-1]
        X_star[1:, :] = X[1:, :] - rho * X[:-1, :]

        # Reestimação OLS nos dados transformados
        pw_model = sm.OLS(y_star, X_star).fit()
        # Resíduos na escala original para próxima iteração
        resid = y - X @ pw_model.params

        if abs(rho - rho_prev) < tol:
            break

    dw_final = durbin_watson(pw_model.resid)
    return pw_model, rho, dw_inicial, dw_final

In [ ]:
# ---------------------------
# 3. Agregação em Série Temporal (Geral e por Raça do Touro)
# ---------------------------
# Série geral (média de todos os 24 animais por data)
serie_geral = (
    dados.groupby(["Datad", "Datad_num"], as_index=False)["Peso"]
    .mean()
    .sort_values("Datad_num")
)
serie_geral["Grupo"] = "Geral (Todos)"

# Séries estratificadas por Raça do Touro
series_racas = (
    dados.groupby(["Raca_touro", "Datad", "Datad_num"], as_index=False)["Peso"]
    .mean()
    .sort_values(["Raca_touro", "Datad_num"])
    .rename(columns={"Raca_touro": "Grupo"})
)

df_series = pd.concat([serie_geral, series_racas], ignore_index=True)

In [ ]:
# ---------------------------
# 4. Execução dos Modelos Prais-Winsten e Cálculo de APC
# ---------------------------
resultados = []
predicoes = []

for grupo, df_g in df_series.groupby("Grupo", sort=False):
    df_g = df_g.sort_values("Datad_num").copy()

    # Modelo Linear: Peso ~ Tempo (ganho médio em kg/dia)
    X = sm.add_constant(df_g["Datad_num"])
    y = df_g["Peso"]
    modelo_lin, rho, dw_ini, dw_fim = prais_winsten(y, X)

    beta1 = modelo_lin.params[1]
    se_beta1 = modelo_lin.bse[1]
    p_val = modelo_lin.pvalues[1]
    ci_low, ci_high = modelo_lin.conf_int()[1]

    # Modelo Log-Linear: log10(Peso) ~ Tempo (para cálculo de APC - Taxa de Variação %)
    # Convertendo tempo para anos (Datad_num / 365.25) para APC anual
    df_g["Anos"] = df_g["Datad_num"] / 365.25
    X_log = sm.add_constant(df_g["Anos"])
    y_log = np.log10(df_g["Peso"])
    modelo_log, _, _, _ = prais_winsten(y_log, X_log)

    b_log = modelo_log.params[1]
    ci_log_low, ci_log_high = modelo_log.conf_int()[1]

    apc = (-1 + 10**b_log) * 100
    apc_low = (-1 + 10**ci_log_low) * 100
    apc_high = (-1 + 10**ci_log_high) * 100

    # Tendência qualitativa
    if p_val < 0.05:
        tendencia = "Crescente" if beta1 > 0 else "Decrescente"
    else:
        tendencia = "Estacionária (p >= 0.05)"

    resultados.append({
        "Grupo": grupo,
        "N_datas": len(df_g),
        "Beta1 (kg/dia)": round(beta1, 4),
        "IC95%_Beta1": f"[{ci_low:.4f}; {ci_high:.4f}]",
        "p-valor": round(p_val, 5),
        "APC_anual (%)": round(apc, 2),
        "IC95%_APC (%)": f"[{apc_low:.2f}%; {apc_high:.2f}%]",
        "Rho (AR1)": round(rho, 3),
        "DW_OLS": round(dw_ini, 3),
        "DW_PW": round(dw_fim, 3),
        "Tendência": tendencia
    })

    # Guardar valores preditos para visualização
    df_g["Peso_Predito_PW"] = modelo_lin.params[0] + beta1 * df_g["Datad_num"]
    predicoes.append(df_g)

df_resultados = pd.DataFrame(resultados)
df_predicoes = pd.concat(predicoes, ignore_index=True)

print("=== RESULTADOS DA REGRESSÃO DE PRAIS-WINSTEN ===")
print(df_resultados.to_string(index=False))

In [ ]:
# ---------------------------
# 5. Visualização das Tendências Ajustadas (Prais-Winsten) com APC na legenda
# ---------------------------
plt.figure(figsize=(10, 6))

# Filtrar apenas as 3 raças para comparação visual limpa
df_plot = df_predicoes[df_predicoes["Grupo"] != "Geral (Todos)"].copy()

# Criar dicionário mapeando cada Grupo ao seu texto formatado com APC e p-valor
mapa_legendas = {}
for _, row in df_resultados[df_resultados["Grupo"] != "Geral (Todos)"].iterrows():
    grp = row["Grupo"]
    apc = row["APC_anual (%)"]
    ic_apc = row["IC95%_APC (%)"]
    pval = row["p-valor"]
    sig = "*" if pval < 0.05 else "ns"
    mapa_legendas[grp] = f"{grp} | APC: {apc:+.2f}% {ic_apc} (p={pval:.3f}, {sig})"

# Aplicar o rótulo enriquecido ao dataframe de plotagem
df_plot["Legenda_APC"] = df_plot["Grupo"].map(mapa_legendas)

# Plotar pontos observados e linhas de tendência ajustadas
sns.scatterplot(
    data=df_plot, x="Datad_num", y="Peso", hue="Legenda_APC", s=80, alpha=0.85
)
sns.lineplot(
    data=df_plot, x="Datad_num", y="Peso_Predito_PW", hue="Legenda_APC",
    linestyle="--", lw=2, legend=False
)

plt.title("Tendência Temporal do Peso por Raça do Touro (Regressão Prais-Winsten)")
plt.xlabel("Dias desde a primeira pesagem (Datad_num)")
plt.ylabel("Peso Médio (kg)")
plt.grid(True, alpha=0.3)
plt.legend(title="Raça do Touro | Taxa de Variação Anual (IC 95%)", loc="upper left", frameon=True)
plt.tight_layout()
plt.savefig("prais_winsten_tendencias_apc.png", dpi=300)
plt.show()